# 00｜先看真实案例和环境

这套课面向第一次使用 Python 图像处理的人。先观察照片，再运行一个小算法，最后检查算法在哪些条件下失效。每本都能单独从上到下执行，建议先读本课。

**操作：**在 JupyterLab 打开笔记本，选中代码格按 `Shift + Enter`。第一次从第一个代码格开始。变量保存当前笔记本的图像，下一格会继续使用；重启内核后应重新逐格运行。

学完本课应能区分：真实照片、原始标注、算法输出、人工真值、派生图。我们使用三个任务：焊缝外观、金属表面缺陷、真实零件数量。没有把它们伪装成刀片电池的七格数据。

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_utils.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import lab_utils as lab
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

## 1. 确认数据没有被换掉

读图之前核对 SHA256。它能发现文件变化，但不能证明标注正确或不同照片来自不同零件。下载器只取少量公开样本，完整工业档案保存在被忽略的缓存中。

In [ ]:
rows = lab.manifest()["files"]
image_rows = [row for row in rows if row["kind"] == "image"]
for row in image_rows:
    image = lab.load_case(row["id"])
    print(row["id"], image.shape, row["license"], row["sha256"][:16])
assert len({row["dataset"] for row in image_rows}) >= 3
assert not lab.manifest()["private_images_included"]

## 2. 看三类真实照片

焊缝照片研究局部纹理；Kolektor 照片研究表面裂纹；螺钉照片研究数量。反光、边界、尺度不同，同一套阈值不应直接照搬。分散零件原作者说明有六颗螺钉和两颗金属球，这里把“所有可见零件”作为计数口径。

In [ ]:
lab.show_images([lab.load_case("weld_good"), lab.load_case("kolektor_defect_0"),
                 lab.load_case("parts_separated")],
                ["真实焊缝：CC0 来源语义", "真实工业表面：原始缺陷掩膜", "真实零件：6 螺钉 + 2 球"], cols=3)

## 3. 标注不是算法预测

下面黄色框来自公开焊缝数据的原始 YOLO 标注；本课没有训练或运行检测器。显示原标注只是为了理解框坐标。YOLO 的中心坐标和宽高以整图归一化；显示时要乘原图宽高，不能固定乘 640。

In [ ]:
image = lab.load_case("weld_good")
labels = lab.pixel_yolo_labels("weld_good", image)
display(labels)
lab.show_images([image, lab.draw_boxes(image, labels)], ["原始实拍", "上游标注叠框（不是推理）"], cols=2)

## 动手检查

把 `weld_good` 改成 `weld_crack`，比较标注。再看图片的 `shape`：顺序是高、宽、通道。OpenCV 读入彩色图片用 BGR 顺序，本课显示函数会转换成 RGB。若发现“数量正确但框错了”，也不能算定位完成。

产线迁移顺序：拍清楚 → 定位汇流排 → 找镍片 → 对齐焊格 → 判七格外观 → 保存证据。强度与内部熔合不能仅凭二维外观保证。

## 来源与练习边界

公开焊缝：sukmaadhiwijaya / rikkarth，固定提交 `8b80d1afe88c0b946500a498cbdb9d7a1eda5533`，数据卡声明 CC0-1.0；工业表面：Kolektor Group / ViCoS，CC BY-NC-SA 4.0；分散零件：Tung1213，CC BY-SA 3.0；重叠零件：Technologyfelix，CC BY-SA 4.0。相关照片和派生图保留原许可及署名。

完整来源、原许可页、下载路径及 SHA256 见 [DATA_SOURCES.md](../DATA_SOURCES.md) 和 [source_manifest.json](../sources/source_manifest.json)。本课是非商业教学。灰度、二值、叠框及扰动均为原照片的派生图，七格合成图另有醒目标记。它们不等于新增独立验收数据，也不证明电池焊接准确率或机械强度。